# Offline evaluation on `shop.history_sessions`

**Copia extendida con el ALS Implicit Collaborative Filtering**

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [2]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0366


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_fe789197d6d2ea15,2026-11-14 02:30:26.819000000,u_7cfa6265,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_c4a35d', 'ts': '2026-11-14 ...","[p_4ea89f, p_966aee, p_cbcc44, p_69d40e, p_8a8...","[0.020482, 0.014404, 0.079019, 0.507986, 0.132...",NaN,NaN
1,r_7d1c3ebd69c7bd41,2026-11-08 03:57:37.978000000,u_2ccc356a,"{'device': 'ios', 'page': 'home', 'region': 'C...",[],"[p_1bcbe1, p_2cc7f6, p_53e599, p_6d0f23, p_b10...","[0.168646, 0.03917, 0.000528, 0.026882, 0.0034...",NaN,NaN
2,r_f594ac4a476350a5,2026-11-12 16:20:10.778000000,u_32b25e40,"{'device': 'ios', 'page': 'home', 'region': 'C...",[],"[p_2aa88b, p_4ba423, p_95d5df, p_49d2c3, p_2c8...","[0.002777, 0.015233, 0.012389, 0.010134, 0.230...",NaN,NaN


In [3]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [4]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [5]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [6]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0106


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.400821,0.006365,0.044027,3884.0
coview,0.234473,0.007959,0.011300,5442.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

___

## Cosas de Ángel

Métricas:
- hit_rate@10 = (# clicked products in top 10) / (# observed clicks)
- IPC_ctr_slot1
- SNIPS_ctr_slot1
- slot1_matches: # observations where the new policy's first recommendation matches the logging policy's first recommendation

In [7]:
LIMIT = 1000000

In [8]:
interactions = sql(f"""
    SELECT
        user_id::String AS user_id,
        product_id::String AS product_id,
        count() AS n
    FROM shop.history_events
    WHERE ts < '{CUT}'
      AND event_type = 'view'
      AND user_id IS NOT NULL
      AND product_id IS NOT NULL
    GROUP BY user_id, product_id
    LIMIT {LIMIT}
""")

print(len(interactions), "user-product interactions")
print(interactions.head())

1000000 user-product interactions
      user_id product_id  n
0  u_f59ded89   p_7e67f4  1
1  u_c59b25de   p_e2803e  1
2  u_9f7d47a5   p_885d5a  1
3  u_4b519c90   p_7c991f  1
4  u_cfec5c65   p_10860f  1


In [9]:
from scipy.sparse import coo_matrix

# Stable integer mappings
users = interactions.user_id.unique()
products = interactions.product_id.unique()

user_to_idx = {u: i for i, u in enumerate(users)}
product_to_idx = {p: i for i, p in enumerate(products)}

idx_to_product = {i: p for p, i in product_to_idx.items()}

user_idx = interactions.user_id.map(user_to_idx).to_numpy()
item_idx = interactions.product_id.map(product_to_idx).to_numpy()

# User x item matrix
user_items = coo_matrix(
    (
        interactions.n.to_numpy(dtype=np.float32),
        (user_idx, item_idx),
    ),
    shape=(len(users), len(products)),
).tocsr()

# Item x user matrix: required by implicit ALS
item_users = user_items.T.tocsr()

print("Users:", user_items.shape[0])
print("Products:", user_items.shape[1])
print("Nonzero interactions:", user_items.nnz)
print("Sparsity:", 1 - user_items.nnz / np.prod(user_items.shape))

Users: 58308
Products: 5000
Nonzero interactions: 1000000
Sparsity: 0.9965699389449132


In [10]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=20,
    alpha=20,
    random_state=42,
)

als.fit(user_items)

print("ALS trained")

/home/angel/Programacion/Datathon6/venv_datathon/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

ALS trained


In [11]:
def als_recommend(user_id, N=10):
    user_id = str(user_id)

    # Cold-start fallback
    if user_id not in user_to_idx:
        return None

    uid = user_to_idx[user_id]

    # No interaction history
    if user_items[uid].nnz == 0:
        return None

    item_ids, scores = als.recommend(
        uid,
        user_items[uid],
        N=N,
        filter_already_liked_items=True,
    )

    return [idx_to_product[int(i)] for i in item_ids]

In [12]:
for u in list(users[:5]):
    print(u, als_recommend(u))

u_f59ded89 ['p_ec8371', 'p_b9a1ca', 'p_1f2b64', 'p_059a74', 'p_9d46dd', 'p_bbd173', 'p_276b63', 'p_e6337c', 'p_817f12', 'p_d94e20']
u_c59b25de ['p_ebf2f0', 'p_0c3c0c', 'p_32b3d1', 'p_cf300e', 'p_98e85d', 'p_8dcf72', 'p_8c4d1c', 'p_f238af', 'p_d39a96', 'p_17a302']
u_9f7d47a5 ['p_67a411', 'p_5733ef', 'p_c74b01', 'p_e0b010', 'p_bd32e4', 'p_4743de', 'p_c30e88', 'p_307c58', 'p_c98f7f', 'p_8e0176']
u_4b519c90 ['p_0de352', 'p_bccc50', 'p_f60d1b', 'p_66f831', 'p_d3f4fd', 'p_7f9b4a', 'p_657a9f', 'p_72285d', 'p_a9b2bd', 'p_68b1c5']
u_cfec5c65 ['p_36fd89', 'p_961097', 'p_4ceff0', 'p_f67b9c', 'p_3378f7', 'p_ed3a9d', 'p_f71b5a', 'p_940300', 'p_a63a60', 'p_966aee']


In [13]:
print("Number of products:", len(product_to_idx))
print("ALS item factors:", als.item_factors.shape)
print("ALS user factors:", als.user_factors.shape)
print("Mapping max index:", max(idx_to_product.keys()))

Number of products: 5000
ALS item factors: (5000, 64)
ALS user factors: (58308, 64)
Mapping max index: 4999


In [14]:
len(product_to_idx) == als.item_factors.shape[0]

True

In [15]:
def als_policy(row):
    recs = als_recommend(row.user_id, N=10)

    if recs is None:
        return baseline(row)

    # Fill any missing slots using the baseline
    recs = list(recs)

    for p in baseline(row):
        if p not in recs:
            recs.append(p)

    return recs[:10]


POLICIES = {
    "baseline": baseline,
    "coview": coview,
    "als": als_policy,
}

### Evaluation (again)

In [16]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0106


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.400821,0.006365,0.044027,3884.0
coview,0.234473,0.007959,0.011300,5442.0
als,0.027360,0.000882,0.003970,430.0


In [17]:
for name, policy in POLICIES.items():
    recs = [
        policy(r)
        for r in test.head(5000).itertuples()
    ]

    unique_products = len({
        p for row in recs for p in row
    })

    print(name, "Unique recommended products:", unique_products)

baseline Unique recommended products: 10
coview Unique recommended products: 1443
als Unique recommended products: 2681


In [18]:
for name, policy in POLICIES.items():
    recs = [
        policy(r)
        for r in test.head(5000).itertuples()
    ]

    recommended = [
        p for row in recs for p in row
    ]

    popularity = pd.Series(recommended).map(
        interactions.groupby("product_id")["n"].sum()
    )

    print(
        name,
        "Average popularity:",
        popularity.mean(),
        "Missing popularity:",
        popularity.isna().mean()
    )

baseline Average popularity: 1567.3 Missing popularity: 0.0
coview Average popularity: 792.73532 Missing popularity: 0.0
als Average popularity: 499.27926 Missing popularity: 0.0


In [19]:
for u in list(users[:5]):
    uid = user_to_idx[u]

    recs = als_recommend(u, N=10)

    history = user_items[uid]

    print("\nUser:", u)
    print("History size:", history.nnz)
    print("Recommendations:", recs)


User: u_f59ded89
History size: 89
Recommendations: ['p_ec8371', 'p_b9a1ca', 'p_1f2b64', 'p_059a74', 'p_9d46dd', 'p_bbd173', 'p_276b63', 'p_e6337c', 'p_817f12', 'p_d94e20']

User: u_c59b25de
History size: 19
Recommendations: ['p_ebf2f0', 'p_0c3c0c', 'p_32b3d1', 'p_cf300e', 'p_98e85d', 'p_8dcf72', 'p_8c4d1c', 'p_f238af', 'p_d39a96', 'p_17a302']

User: u_9f7d47a5
History size: 13
Recommendations: ['p_67a411', 'p_5733ef', 'p_c74b01', 'p_e0b010', 'p_bd32e4', 'p_4743de', 'p_c30e88', 'p_307c58', 'p_c98f7f', 'p_8e0176']

User: u_4b519c90
History size: 44
Recommendations: ['p_0de352', 'p_bccc50', 'p_f60d1b', 'p_66f831', 'p_d3f4fd', 'p_7f9b4a', 'p_657a9f', 'p_72285d', 'p_a9b2bd', 'p_68b1c5']

User: u_cfec5c65
History size: 18
Recommendations: ['p_36fd89', 'p_961097', 'p_4ceff0', 'p_f67b9c', 'p_3378f7', 'p_ed3a9d', 'p_f71b5a', 'p_940300', 'p_a63a60', 'p_966aee']


In [20]:
print("Training users:", user_items.shape[0])
print("Training products:", user_items.shape[1])
print("Training interactions:", user_items.nnz)

print("\nInteractions per user:")
print(
    np.percentile(
        np.diff(user_items.indptr),
        [0, 25, 50, 75, 90, 99, 100]
    )
)

Training users: 58308
Training products: 5000
Training interactions: 1000000

Interactions per user:
[  1.   6.  11.  22.  38.  91. 446.]


In [21]:
# Retrieve up to 10 million aggregated interactions in batches

BATCH_SIZE = 1_000_000
MAX_ROWS = 10_000_000

parts = []

for offset in range(0, MAX_ROWS, BATCH_SIZE):

    print(f"Fetching batch: {offset:,} - {offset + BATCH_SIZE:,}")

    batch = sql(f"""
        SELECT
            user_id::String AS user_id,
            product_id::String AS product_id,
            count() AS n
        FROM shop.history_events
        WHERE ts < '{CUT}'
          AND event_type = 'view'
          AND user_id IS NOT NULL
          AND product_id IS NOT NULL
        GROUP BY user_id, product_id
        ORDER BY user_id, product_id
        LIMIT {BATCH_SIZE} OFFSET {offset}
    """)

    if len(batch) == 0:
        print("No more interactions.")
        break

    parts.append(batch)

    print(f"Retrieved: {len(batch):,}")

    if len(batch) < BATCH_SIZE:
        print("Last batch reached.")
        break

# Combine all batches
interactions = pd.concat(parts, ignore_index=True)

print("\nFinal dataset:")
print("User-product pairs:", f"{len(interactions):,}")
print("Total views:", f"{interactions.n.sum():,}")
print("Unique users:", f"{interactions.user_id.nunique():,}")
print("Unique products:", f"{interactions.product_id.nunique():,}")


Fetching batch: 0 - 1,000,000
Retrieved: 1,000,000
Fetching batch: 1,000,000 - 2,000,000
Retrieved: 1,000,000
Fetching batch: 2,000,000 - 3,000,000
Retrieved: 1,000,000
Fetching batch: 3,000,000 - 4,000,000
Retrieved: 1,000,000
Fetching batch: 4,000,000 - 5,000,000
Retrieved: 1,000,000
Fetching batch: 5,000,000 - 6,000,000
Retrieved: 1,000,000
Fetching batch: 6,000,000 - 7,000,000
Retrieved: 1,000,000
Fetching batch: 7,000,000 - 8,000,000
Retrieved: 793,783
Last batch reached.

Final dataset:
User-product pairs: 7,793,783
Total views: 9,069,793
Unique users: 59,771
Unique products: 5,000


In [22]:
from scipy.sparse import coo_matrix

users = interactions.user_id.unique()
products = interactions.product_id.unique()

user_to_idx = {u: i for i, u in enumerate(users)}
product_to_idx = {p: i for i, p in enumerate(products)}

idx_to_product = {i: p for p, i in product_to_idx.items()}

user_idx = interactions.user_id.map(user_to_idx).to_numpy()
item_idx = interactions.product_id.map(product_to_idx).to_numpy()

user_items = coo_matrix(
    (
        interactions.n.to_numpy(dtype=np.float32),
        (user_idx, item_idx)
    ),
    shape=(len(users), len(products))
).tocsr()

item_users = user_items.T.tocsr()

print("Users:", user_items.shape[0])
print("Products:", user_items.shape[1])
print("Interactions:", user_items.nnz)
print("Sparsity:", 1 - user_items.nnz / np.prod(user_items.shape))


Users: 59771
Products: 5000
Interactions: 7793783
Sparsity: 0.9739211892054676


In [23]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=20,
    alpha=20,
    random_state=42
)

als.fit(user_items)

print("ALS trained")
print("User factors:", als.user_factors.shape)
print("Item factors:", als.item_factors.shape)


  0%|          | 0/20 [00:00<?, ?it/s]

ALS trained
User factors: (59771, 64)
Item factors: (5000, 64)


In [24]:
print("Interactions per user:")
print(
    interactions.groupby("user_id")["n"].sum().describe(
        percentiles=[.25, .5, .75, .9, .99]
    )
)

print("\nInteractions per product:")
print(
    interactions.groupby("product_id")["n"].sum().describe(
        percentiles=[.25, .5, .75, .9, .99]
    )
)

Interactions per user:
count    59771.000000
mean       151.742367
std        199.606510
min          2.000000
25%         45.000000
50%         92.000000
75%        183.000000
90%        333.000000
99%        953.300000
max       7467.000000
Name: n, dtype: float64

Interactions per product:
count     5000.000000
mean      1813.958600
std       1405.191867
min         94.000000
25%        926.500000
50%       1429.000000
75%       2235.000000
90%       3427.300000
99%       7485.580000
max      16841.000000
Name: n, dtype: float64


In [29]:
def als_policy(row):
    recs = als_recommend(row.user_id, N=10)

    if recs is None:
        return baseline(row)

    # Fill any missing slots using the baseline
    recs = list(recs)

    for p in baseline(row):
        if p not in recs:
            recs.append(p)

    return recs[:10]


POLICIES = {
    "baseline": baseline,
    "coview": coview,
    "als": als_policy,
}

In [30]:
POLICIES = {
    "baseline": baseline,
    "coview": coview,
    "als": als_policy,
}

In [31]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0106


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.400821,0.006365,0.044027,3884.0
coview,0.234473,0.007959,0.011300,5442.0
als,0.047332,0.003467,0.014151,347.0
